# Vessel Ops AI — Unsloth Fine-tune: Gemma 4 × WHO IMGS

Fine-tunes `gemma-4-2b` on a Q&A dataset derived from the WHO *International Medical Guide for Ships* (3rd Edition) using Unsloth + QLoRA.

**Kaggle runtime:** T4 GPU (16 GB VRAM) · ~2–3 hours to train

**Output:** LoRA adapter + merged GGUF pushed to HuggingFace Hub for use with Ollama

---
**Tracks targeted:** Unsloth Special Technology Prize · Health & Sciences · Global Resilience

In [1]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers>=4.51.3,<=5.5.0",
    "--force-reinstall"
], check=True)
# Restart kernel after this cell if you haven't already

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2026.4.0 which is incompatible.
datasets 4.8.3 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
sigstore 4.2.0 requires rich<15,>=13, but you have rich 15.0.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you ha

CompletedProcess(args=['pip', 'install', '-q', 'transformers>=4.51.3,<=5.5.0', '--force-reinstall'], returncode=0)

In [2]:
import os
for item in os.listdir("/kaggle/input/datasets/nswitzer/who-international-medical-guide-for-ships-3rd-ed"):
    print(item)

WHO International Medical Guide for Ships.pdf
who_imgs.json
who_imgs_qa.jsonl


In [3]:
# Install dependencies (Kaggle GPU runtime)
!pip install unsloth datasets huggingface_hub -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [4]:
import json, os
from pathlib import Path
from datasets import Dataset
from huggingface_hub import login

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID       = "unsloth/gemma-4-E2B-it"          # Unsloth-optimised Gemma 4 2B instruct
LORA_RANK      = 16
MAX_SEQ_LEN    = 2048
BATCH_SIZE     = 2
GRAD_ACCUM     = 4                              # effective batch = 8
EPOCHS         = 3
LR             = 2e-4
OUTPUT_DIR     = "/kaggle/working/vessel-ops-gemma4"
HF_REPO_ID     = "vessel-ops-ai/gemma4-maritime-medical"  # update to your HF org
DATASET_PATH = "/kaggle/input/datasets/nswitzer/who-international-medical-guide-for-ships-3rd-ed/who_imgs_qa.jsonl"  # upload as Kaggle dataset

# HuggingFace token — add as a Kaggle Secret named HF_TOKEN
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [5]:
# ── Load dataset ──────────────────────────────────────────────────────────────
records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} training examples")
print("Sample:")
sample = records[42]["conversations"]
for turn in sample:
    print(f"  [{turn['from']}]: {turn['value'][:120]}...")

Loaded 903 training examples
Sample:
  [system]: You are an expert maritime medicine assistant trained on the WHO International Medical Guide for Ships (3rd Edition). Yo...
  [human]: What does the WHO IMGS recommend regarding: ●
force your fist upwards into your upper ab domen?...
  [gpt]: According to the WHO International Medical Guide for Ships (p.31):

●
force your fist upwards into your upper ab domen.
...


In [6]:
# ── Load model with Unsloth ───────────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,
)
print(model.print_trainable_parameters())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


RuntimeError: ***** numpy was upgraded mid-session (loaded: 2.0.2, installed: 2.4.5) but the kernel still has the old version in memory. numpy uses C extensions that cannot be reloaded without restarting. Please restart your runtime/kernel after installing packages. *****

In [ ]:
# ── Format conversations into Gemma chat template ─────────────────────────────
def format_conversation(example):
    convs = example["conversations"]
    # Build a list of {role, content} dicts
    messages = []
    for turn in convs:
        role_map = {"system": "system", "human": "user", "gpt": "assistant"}
        role = role_map.get(turn["from"], turn["from"])
        messages.append({"role": role, "content": turn["value"]})
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

raw_dataset = Dataset.from_list(records)
dataset = raw_dataset.map(format_conversation, remove_columns=raw_dataset.column_names)
dataset = dataset.filter(lambda x: len(x["text"]) < MAX_SEQ_LEN * 4)  # rough char filter

# 90/10 train/val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=True,                    # Unsloth packing for efficiency
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Peak VRAM: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_questions = [
    "A crew member has severe chest pain radiating to his left arm. He is sweating and pale. What should I do?",
    "How do I treat a deep laceration that won't stop bleeding?",
    "A sailor has symptoms of appendicitis 400 miles from shore. What are the signs and what can I give them?",
]

SYSTEM = (
    "You are an expert maritime medicine assistant trained on the WHO International "
    "Medical Guide for Ships (3rd Edition). You provide accurate, concise medical "
    "guidance to ship officers managing emergencies at sea with no doctor available. "
    "Always cite the relevant WHO IMGS page number when referencing specific protocols."
)

for q in test_questions:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "text", "text": q}]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True          # ← add return_dict=True
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],        # ← pass mask explicitly
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],      # ← index via dict key
        skip_special_tokens=True
    )
    print(f"Q: {q}\nA: {response}\n{'='*60}")

In [ ]:
from huggingface_hub import whoami
info = whoami(token=HF_TOKEN)
print("User:", info["name"])
print("Orgs:", [o["name"] for o in info.get("orgs", [])])

In [ ]:
HF_REPO_ID = "nswitzer/gemma4-maritime-medical"

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

In [ ]:
# ── Save LoRA adapter to HuggingFace Hub ─────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"LoRA adapter pushed to hf.co/{HF_REPO_ID}")

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/vessel-ops-gemma4-merged", ignore_errors=True)

model.push_to_hub_gguf(
    "nswitzer/gemma4-maritime-medical-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)

In [ ]:
# ── Export merged GGUF (Q4_K_M) for Ollama ─────────────────────────────────
# push_to_hub_gguf handles merge + convert + quantise + upload in one call.
GGUF_REPO = HF_REPO_ID + "-GGUF"

print(f"Pushing GGUF to hf.co/{GGUF_REPO} — takes ~16 min, do not interrupt...")

model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method=["q4_k_m"],   # list form — more reliable across Unsloth versions
    token=HF_TOKEN,
)

print(f"\nGGUF model pushed to hf.co/{GGUF_REPO}")
print(f"Install:  ollama pull hf.co/{GGUF_REPO}")
print(f"Test:     ollama run hf.co/{GGUF_REPO}")

In [ ]:
model.push_to_hub_gguf(
    "nswitzer/gemma4-maritime-medical-GGUF",
    tokenizer,
    quantization_method=["q4_k_m"],
    token=HF_TOKEN,
)

In [ ]:
# Cell 1 — reload from HF (skip retraining entirely, ~5 min)
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name="nswitzer/gemma4-maritime-medical",
    max_seq_length=2048,
    load_in_4bit=True,
)

![Push It](https://media1.tenor.com/m/W5qMGPxWbA8AAAAC/salt-n-pepa-push-it.gif)

## Push it to Hugging Face so we can serve it up in Ollama

In [ ]:
model.push_to_hub_gguf(
    "nswitzer/gemma4-maritime-medical-GGUF",
    tokenizer,
    quantization_method=["q4_k_m"],
    token=HF_TOKEN,
)

## Integrating the Fine-tuned Model with Vessel Ops AI

Once the GGUF is on HuggingFace, I'll swap it into the app:

**Ollama (recommended for desktop companion)**
```bash
ollama run hf.co/vessel-ops-ai/gemma4-maritime-medical-GGUF
MODEL_PRIMARY=vessel-ops-ai/gemma4-maritime-medical-GGUF
```

The fine-tuned model internalises the WHO IMGS corpus, so even without the BM25 RAG layer it produces page-cited maritime medical guidance. With RAG enabled on top, hallucination is further reduced.